# Supervisor results summary

A compact, **read-only** presentation of the final thesis results. Tables and figures are generated by `05_results_tables_figures.ipynb`; this notebook does not reanalyse participant data or duplicate statistical values. Run notebooks **01 → 02 → 03 → 04 → 05** first.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Define the expected outputs from Notebook 05.
TABLE_FILES = {
    "Descriptive statistics": "table_01_quantitative_descriptives.csv",
    "RQ1 and RQ2": "table_02_quantitative_results.csv",
    "Final themes and subthemes": "table_03_qualitative_themes.csv",
}

FIGURE_FILES = [
    "figure_01_gad7_distribution.png",
    "figure_02_anxiety_llm_use_relationship.png",
    "figure_03_subtheme_participant_breadth.png",
    "figure_04_final_thematic_map.png",
]

# Locate the project root, regardless of the notebook folder.
cwd = Path.cwd().resolve()

def required_outputs(root):
    return (
        [root / "tables" / name for name in TABLE_FILES.values()]
        + [root / "figures" / name for name in FIGURE_FILES]
    )

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if all(path.is_file() for path in required_outputs(folder))
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate all seven Notebook 05 outputs. "
        f"Searching from: {cwd}. "
        "Check that Notebook 05 completed successfully."
    )

# Define the correct output directories.
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR = PROJECT_ROOT / "figures"

# Verify that all expected files exist.
needed = required_outputs(PROJECT_ROOT)

missing = [
    str(path.relative_to(PROJECT_ROOT))
    for path in needed
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing Notebook 05 outputs:\n"
        + "\n".join(missing)
    )

# Load aggregate results.
tables = {
    label: pd.read_csv(TABLES_DIR / filename)
    for label, filename in TABLE_FILES.items()
}

# Confirm successful loading.
print("Working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)

print(
    "PASS: all three aggregate tables and "
    "four figures are available."
)

## Consistency checks

These checks catch stale or inconsistent outputs. Reported values are checked at the precision used in the thesis; they are **not** substituted into the exported tables.

In [ ]:
desc = tables["Descriptive statistics"]
results = tables["RQ1 and RQ2"]
themes = tables["Final themes and subthemes"]

assert list(desc.columns) == ["Variable", "N", "Mean", "SD", "Median", "Q1", "Q3", "Min", "Max"]
assert list(results.columns) == ["Research question", "Analysis", "Method", "Estimate", "Statistic", "p", "95% CI", "Interpretation"]
assert {"Theme", "Final theme", "Subtheme", "Participants represented", "Coded excerpts"}.issubset(themes.columns)
assert len(desc) == 4 and len(results) == 4 and len(themes) == 8
assert (pd.to_numeric(desc["N"]) == 124).all()

expected_descriptives = {
    "GAD-7 total": (8.00, 5.85),
    "AI self-efficacy mean": (4.99, 0.89),
    "Overall LLM use": (3.45, 1.22),
    "Coding-task LLM use mean": (3.15, 1.08),
}
for label, (mean, sd) in expected_descriptives.items():
    row = desc.loc[desc["Variable"] == label]
    assert len(row) == 1, f"Missing or duplicated descriptive row: {label}"
    assert np.isclose(float(row.iloc[0]["Mean"]), mean, atol=0.0051)
    assert np.isclose(float(row.iloc[0]["SD"]), sd, atol=0.0051)

expected_tests = [
    ("RQ1", "Spearman", 0.1240, 0.1701),
    ("RQ1", "Pearson", 0.1399, 0.1213),
    ("RQ1", "Spearman", 0.1599, 0.0760),
    ("RQ2", "Centered OLS moderation", -0.0102, 0.5682),
]
for (_, row), (rq, method, estimate, p) in zip(results.iterrows(), expected_tests):
    assert row["Research question"] == rq and row["Method"] == method
    assert np.isclose(float(row["Estimate"]), estimate, atol=0.00011)
    assert np.isclose(float(row["p"]), p, atol=0.00011)

assert themes["Subtheme"].nunique() == 8
assert themes["Theme"].nunique() == 4
print("PASS: aggregate values, sample sizes, 4 themes and 8 subthemes agree with expected thesis reporting.")

## Result tables

In [ ]:
for label, table in tables.items():
    print(f"\n{label}")
    display(table)

## Figures

All figures are loaded from Notebook 05 outputs. The subtheme evidence figure represents **participants represented in coded excerpts**, not prevalence estimates. Individual respondents may appear in multiple subthemes.

In [ ]:
for filename in FIGURE_FILES:
    print(filename)
    display(Image(filename=str(FIGURES_DIR / filename)))

## Interpretation for supervisor review

**RQ1:** The associations between reported anxiety symptoms and coding-related LLM-use frequency were small and positive, but not statistically significant for either outcome. Spearman correlations are the emphasized rank-based estimates; Pearson is supplementary.

**RQ2:** The mean-centered anxiety × AI self-efficacy interaction was small and not statistically significant. The bootstrap interval and other model diagnostics are documented in Notebook 03. Non-significance does not establish that an effect is absent.

**Qualitative complement:** Four final themes and eight subthemes describe perceived assistance, reassurance, independent learning and reliance, and trust/verification. Interpretive decisions are documented in Notebook 04; coded counts are not participant prevalence estimates.

**Limitations:** Cross-sectional self-report findings do not establish causal effects or objective changes in mental health, learning, or programming ability.

**Privacy:** This notebook reads only aggregate tables and figures. Do not publicly upload the participant-level survey, Taguette database, quotation audits, or notebooks containing saved private outputs. A separate repository and Git-history audit is still required.

In [ ]:
print("SUPERVISOR SUMMARY COMPLETE")
print("PASS: 3 aggregate tables and 4 figures displayed, with consistency checks passed.")